# Interface Matrix - UNIBE

Sistema de interface estilo The Matrix con 5 canales UNIBE.
Cada dedo levantado activa un nodo con lluvia de caracteres cayendo.

| Dedo | Nodo |
|------|------|
| Pulgar | U |
| Indice | N |
| Medio | I |
| Anular | B |
| Meñique | E |

**Controles:** Levanta dedos para activar canales. ESC para salir.

In [ ]:
!pip install opencv-python mediapipe numpy

In [1]:
import cv2
import mediapipe as mp
import numpy as np
import random
import time
import math

In [2]:
NODOS = [
    {"dedo": "Pulgar",  "letra": "U", "pos_x": 64},
    {"dedo": "Indice",  "letra": "N", "pos_x": 192},
    {"dedo": "Medio",   "letra": "I", "pos_x": 320},
    {"dedo": "Anular",  "letra": "B", "pos_x": 448},
    {"dedo": "Menique", "letra": "E", "pos_x": 576},
]

CHAR_MATRIX = "01アイウエオカキクケコサシスセソタチツテトナニヌネノハヒフヘホマミムメモヤユヨラリルレロワヲン{}[]<>|/\\!@#$%^&*"

NODO_Y = 130
NODO_RADIO = 38
VERDE = (0, 255, 80)
VERDE_OSCURO = (0, 60, 20)

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(static_image_mode=False, max_num_hands=1,
                       min_detection_confidence=0.7, min_tracking_confidence=0.7)
mp_draw = mp.solutions.drawing_utils

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

columnas = 40
lluvia_global = []
for c in range(columnas):
    lluvia_global.append({
        "y": random.randint(-480, 0),
        "velocidad": random.uniform(2.0, 7.0),
        "chars": [random.choice(CHAR_MATRIX) for _ in range(random.randint(8, 25))],
        "x_offset": random.randint(-3, 3)
    })
lluvia_nodos = [[] for _ in range(5)]


def detectar_dedos(hand_landmarks):
    lm = hand_landmarks.landmark
    return [
        1 if lm[4].x < lm[3].x else 0,
        1 if lm[8].y < lm[6].y else 0,
        1 if lm[12].y < lm[10].y else 0,
        1 if lm[16].y < lm[14].y else 0,
        1 if lm[20].y < lm[18].y else 0,
    ]


def dibujar_lluvia_global(frame):
    h, w = frame.shape[:2]
    espaciado = w // columnas
    for c in range(columnas):
        col = lluvia_global[c]
        col["y"] += col["velocidad"]
        x = c * espaciado + espaciado // 2 + col["x_offset"]
        for j, char in enumerate(col["chars"]):
            cy = int(col["y"] + j * 14)
            if -20 < cy < h:
                if j == 0:
                    color = (0, 255, 100)
                elif j < 3:
                    color = (0, 180, 0)
                else:
                    fade = max(30, 200 - j * 12)
                    color = (0, fade, 0)
                if random.random() < 0.03:
                    col["chars"][j] = random.choice(CHAR_MATRIX)
                cv2.putText(frame, char, (x, cy),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
        if col["y"] - len(col["chars"]) * 14 > h:
            col["y"] = random.randint(-200, -40)
            col["velocidad"] = random.uniform(2.0, 7.0)
            col["chars"] = [random.choice(CHAR_MATRIX) for _ in range(random.randint(8, 25))]


def dibujar_lluvia_nodo(frame, idx):
    h, w = frame.shape[:2]
    nodo = NODOS[idx]
    cx = nodo["pos_x"]
    if len(lluvia_nodos[idx]) < 20:
        lluvia_nodos[idx].append({
            "x": cx + random.randint(-25, 25),
            "y": NODO_Y - NODO_RADIO - random.randint(5, 30),
            "velocidad": random.uniform(2.0, 5.0),
            "chars": [random.choice(CHAR_MATRIX) for _ in range(random.randint(5, 15))],
            "alpha": 255
        })
    vivas = []
    for gota in lluvia_nodos[idx]:
        gota["y"] += gota["velocidad"]
        gota["alpha"] = max(0, gota["alpha"] - 2)
        if gota["y"] < h and gota["alpha"] > 10:
            vivas.append(gota)
            for j, char in enumerate(gota["chars"]):
                cy = int(gota["y"] + j * 14)
                cx2 = int(gota["x"])
                if 0 < cy < h and 0 < cx2 < w:
                    a = max(20, gota["alpha"] - j * 25)
                    color = (0, 255, 120) if j == 0 else (0, int(a * 0.8), 0)
                    cv2.putText(frame, char, (cx2, cy),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
    lluvia_nodos[idx] = vivas


def dibujar_nodo(frame, idx, activo, fc):
    h, w = frame.shape[:2]
    nodo = NODOS[idx]
    cx = nodo["pos_x"]
    cy = NODO_Y
    radio = NODO_RADIO
    if activo:
        glow = int(80 + 50 * math.sin(fc * 0.12))
        overlay = frame.copy()
        cv2.circle(overlay, (cx, cy), radio + 25, (0, glow, 0), -1)
        cv2.addWeighted(overlay, 0.3, frame, 0.7, 0, frame)
        cv2.circle(frame, (cx, cy), radio - 5, (0, 40, 10), -1)
        cv2.circle(frame, (cx, cy), radio, VERDE, 3)
        cv2.circle(frame, (cx, cy), radio + 4, (0, 150, 50), 1)
        letra = nodo["letra"]
        ts = cv2.getTextSize(letra, cv2.FONT_HERSHEY_SIMPLEX, 1.2, 3)[0]
        cv2.putText(frame, letra, (cx - ts[0] // 2, cy + ts[1] // 2),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, VERDE, 3)
        us = cv2.getTextSize("UNIBE", cv2.FONT_HERSHEY_SIMPLEX, 0.35, 1)[0]
        cv2.putText(frame, "UNIBE", (cx - us[0] // 2, cy + radio + 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.35, VERDE, 1)
        for ly in range(cy + radio + 28, h - 50, 8):
            if random.random() > 0.3:
                a = max(30, 255 - (ly - cy) * 2)
                cv2.line(frame, (cx, ly), (cx, ly + 4), (0, int(a * 0.6), 0), 1)
        if h > 220:
            modulo = h - 50 - cy - radio
            if modulo > 0:
                py = (cy + radio + 28 + fc * 3) % modulo
                py = int(cy + radio + 28 + py)
                if py < h - 50:
                    cv2.rectangle(frame, (cx - 4, py), (cx + 4, py + 5), VERDE, -1)
    else:
        cv2.circle(frame, (cx, cy), radio, (0, 30, 10), 1)
        letra = nodo["letra"]
        ts = cv2.getTextSize(letra, cv2.FONT_HERSHEY_SIMPLEX, 1.0, 2)[0]
        cv2.putText(frame, letra, (cx - ts[0] // 2, cy + ts[1] // 2),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 35, 12), 2)


def dibujar_conexiones(frame, dedos):
    activos = [(NODOS[i]["pos_x"]) for i in range(5) if dedos[i]]
    if len(activos) < 2:
        return
    for i in range(len(activos) - 1):
        x1, x2 = activos[i], activos[i + 1]
        for px in range(x1 + NODO_RADIO, x2 - NODO_RADIO, 6):
            cv2.line(frame, (px, NODO_Y), (px + 3, NODO_Y), (0, 60, 0), 1)


def dibujar_barra(frame, dedos):
    h, w = frame.shape[:2]
    cv2.rectangle(frame, (0, h - 45), (w, h), (5, 5, 5), -1)
    cv2.line(frame, (0, h - 45), (w, h - 45), (0, 80, 0), 1)
    activos = sum(dedos)
    if activos == 5:
        txt = ">> UNIBE: TODOS LOS NODOS <<"
        c = VERDE
    elif activos > 0:
        letras = " ".join(NODOS[i]["letra"] for i in range(5) if dedos[i])
        txt = f">> UNIBE: {activos}/5 | {letras} <<"
        c = (0, 200, 60)
    else:
        txt = ">> UNIBE: ESPERANDO... <<"
        c = (0, 50, 15)
    cv2.putText(frame, txt, (10, h - 18), cv2.FONT_HERSHEY_SIMPLEX, 0.45, c, 1)
    for i in range(5):
        ix = w - 120 + i * 22
        iy = h - 25
        if dedos[i]:
            cv2.rectangle(frame, (ix, iy), (ix + 14, iy + 14), VERDE, -1)
        else:
            cv2.rectangle(frame, (ix, iy), (ix + 14, iy + 14), (0, 20, 5), -1)
        cv2.rectangle(frame, (ix, iy), (ix + 14, iy + 14), (0, 60, 0), 1)


if not cap.isOpened():
    print("ERROR: No se pudo abrir la camara")
else:
    print("Camara OK - Presiona ESC para salir")
    fc = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        fc += 1
        frame = cv2.flip(frame, 1)
        h, w, _ = frame.shape
        frame = cv2.convertScaleAbs(frame, alpha=0.4, beta=-30)
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(rgb)
        dedos = [0, 0, 0, 0, 0]
        if results.multi_hand_landmarks:
            hand_lm = results.multi_hand_landmarks[0]
            mp_draw.draw_landmarks(frame, hand_lm, mp_hands.HAND_CONNECTIONS)
            dedos = detectar_dedos(hand_lm)
        dibujar_lluvia_global(frame)
        vs = (0, 200, 60)
        cv2.line(frame, (10, 25), (w - 10, 25), (0, 60, 0), 1)
        cv2.putText(frame, ">> THE MATRIX - UNIBE //", (10, 18),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, vs, 1)
        cv2.line(frame, (10, 30), (w - 10, 30), (0, 60, 0), 1)
        dibujar_conexiones(frame, dedos)
        for i in range(5):
            if dedos[i]:
                dibujar_lluvia_nodo(frame, i)
        for i in range(5):
            dibujar_nodo(frame, i, dedos[i], fc)
        dibujar_barra(frame, dedos)
        cv2.putText(frame, "ESC", (w - 40, 18),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 50, 0), 1)
        cv2.imshow("Matrix UNIBE", frame)
        key = cv2.waitKey(1) & 0xFF
        if key == 27:
            break
        try:
            if cv2.getWindowProperty("Matrix UNIBE", cv2.WND_PROP_VISIBLE) < 1:
                break
        except:
            break
    cap.release()
    cv2.destroyAllWindows()
    print("Sesion terminada")

Camara OK - Presiona ESC para salir


c:\Users\fanny\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Sesion terminada
